## 앙상블 검색 
- 희소 검색과 밀집 검색을 함께 쓰는 접근 방식이다. 
- 희소 검색은 빠르고 직관적인 검색이 가능하지만 문맥과 의미를 파악하는 데에 한계가 존재한다. 
- 밀집 검색은 텍스트의 의미를 깊이 있게 이해하고 관련성을 판단가능하지만 계산 비용이 높고 특정 표현을 정확하게 포착하는데 어려움이 있다. 

In [6]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = "./data/투자설명서.pdf"
loader = PyPDFLoader(file_path)


doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200) 

docs = loader.load_and_split(doc_splitter)

In [2]:
from langchain_community.retrievers import BM25Retriever 
from kiwipiepy import Kiwi 

kiwi_tokenizer = Kiwi()

def kiwi_tokenize(text):
    return [token.form for token in kiwi_tokenizer.tokenize(text)]

In [3]:
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 4

In [12]:
from langchain_openai.embeddings import OpenAIEmbeddings 

embedding = OpenAIEmbeddings(
            model="bge-m3",
            base_url="http://localhost:1234/v1",
            api_key="lm-studio",
            check_embedding_ctx_length=False)


In [13]:
print(len(docs))
print(type(docs))
print(type(docs[0]))
print(type(docs[0].page_content))
print(repr(docs[0].page_content[:100]))

522
<class 'list'>
<class 'langchain_core.documents.base.Document'>
<class 'str'>
'투 자 설 명 서  \n \n  \n2024년 \xa0 \xa006월 \xa0 26일\n\xa0\n( 발 \xa0 \xa0 행 \xa0 \xa0 회 \xa0 \xa0 사 \xa0 \xa0명 )\n주식회사 셀리드\n( 증권의 종목과 발행증권수 )\n기명식 보통'


In [14]:
from langchain_community.vectorstores import FAISS 

faiss_store = FAISS.from_documents(docs, embedding)
faiss_store.save_local("./content/DB")

In [15]:
persist_directory = "./content/DB" 
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

In [16]:
faiss_retriever = vectordb.as_retriever(search_kwargs={"k" : 4}) 


In [20]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble = EnsembleRetriever(retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5])


In [21]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

system_prompt = """
당신은 문서 기반 질의응답 assistant입니다.
반드시 아래 context를 기반으로만 답변하세요.
모르면 모른다고 답하세요.

context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
)

qa_chain = create_retrieval_chain(
    retriever=ensemble,
    combine_docs_chain=question_answer_chain,
)

In [ ]:
qa_chain.invoke({"input" : "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"})